In [ ]:
import sys
import pickle
from codebot.tokenizer import train_bpe

sys.path.append(".")

vocab_size = 1000
text = open("codebot/tiny_codes.txt", encoding = "utf-8").read()
merge_rules = train_bpe(text, vocab_size)

with open("codebot/merge_rules.pkl", "wb") as f:
    pickle.dump(merge_rules, f)

In [ ]:
import sys
from codebot.tokenizer import BPETokenizer

sys.path.append(".")

tokenizer = BPETokenizer.load_from("codebot/merge_rules.pkl")

print("처음 학습된 10개")
for token_id in range(256, 266):
    byte_seq = tokenizer.id_to_bytes[token_id]
    text = byte_seq.decode("utf-8", errors = "replace")
    print(token_id, text)

print("마지막 학습된 10개")
for token_id in range(990, 1000):
    byte_seq = tokenizer.id_to_bytes[token_id]
    text = byte_seq.decode("utf-8", errors = "replace")
    print(token_id, text)

sample_text = open("codebot/tiny_codes.txt", encoding = "utf-8").read()[:10000]

byte_count = len(sample_text.encode("utf-8"))
ids = tokenizer.encode(sample_text)
ids_count = len(ids)
compression_ratio = byte_count / ids_count

print("바이트수 :", byte_count)
print("토큰수 :", ids_count)
print("압축률 :", compression_ratio)

In [ ]:
import sys
import numpy as np
from codebot.tokenizer import BPETokenizer

sys.path.append(".")

tokenizer = BPETokenizer.load_from("codebot/merge_rules.pkl")

text = open("codebot/tiny_codes.txt", encoding = "utf-8").read()
ids = tokenizer.encode(text, show_progress = True)

ids_array = np.array(ids, dtype = np.uint16)
ids_array.tofile("codebot/tiny_codes.bin")

print(len(ids_array))
print(ids_array[:20])

In [ ]:
import numpy as np
import sys
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
import matplotlib.pyplot as plt
from itertools import cycle
from tqdm import tqdm

from codebot.model import GPT
from codebot.utils import get_device

sys.path.append(".")

device = get_device()
data_path = "codebot/tiny_codes.bin"
tokenizer_path = "codebot/merge_rules.pkl"
model_save_path = "codebot/model_pretrain.pt"

context_len = 256
vocab_size = 1000
batch_size = 32
learning_rate = 3e-4

max_iters = 20000
embed_dim = 384
n_head = 6
n_layer = 6
ff_dim = 4 * embed_dim
dropout_rate = 0.1

In [ ]:
class TokenDataset(Dataset):
    def __init__(self, tokens, context_len):
        self.tokens = torch.tensor(tokens, dtype = torch.long)
        self.context_len = context_len

    def __len__(self):
        return len(self.tokens) - self.context_len

    def __getitem__(self, idx):
        x = self.tokens[idx:idx + self.context_len]
        y = self.tokens[idx + 1: idx + self.context_len + 1]
        return x, y

In [ ]:
ids = np.fromfile(data_path, dtype = np.uint16)
dataset = TokenDataset(ids, context_len)
dataloader = DataLoader(dataset, batch_size = batch_size, shuffle = True)

model = GPT(vocab_size = vocab_size,
            max_context_len = context_len,
            embed_dim = embed_dim,
            n_head = n_head,
            n_layer = n_layer,
            ff_dim = ff_dim,
            dropout_rate = dropout_rate).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

total_params = sum(p.numel() for p in model.parameters())
print("파라미터수 :", total_params)

losses = []
data_iter = cycle(dataloader)
pbar = tqdm(range(max_iters))

In [ ]:
pbar = tqdm(range(max_iters))
for i in pbar:
    batch_x, batch_y = next(data_iter)
    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    logits = model(batch_x)

    loss = F.cross_entropy(logits.view(-1, logits.size(-1)), batch_y.view(-1))

    optimizer.zero_grad()
    loss.backward()

    optimizer.step()

    losses.append(loss.item())
    pbar.set_postfix({"loss": f"{loss.item():.4f}"})

plt.figure(figsize = (10, 6))
plt.plot(losses)
plt.xlabel("반복")
plt.ylabel("손실")
plt.grid(True)
plt.savefig("loss_pretrain.png")

model.save(model_save_path)

In [ ]:
import sys
sys.path.append(".")

import torch
import torch.nn.functional as F
from codebot.model import GPT
from codebot.tokenizer import BPETokenizer
from codebot.utils import get_device
from codebot.utils import generate

device = get_device()
print(device)

model_save_path = "codebot/model_pretrain.pt"
tokenizer_path = "codebot/merge_rules.pkl"

prompt = "def"
max_new_tokens = 200
temperature = 1.0

tokenizer = BPETokenizer.load_from(tokenizer_path)
model = GPT.load_from(model_save_path, device = device)

In [ ]:
for i in range(5):
    print(i)
    generated_text = generate(model = model,
                              tokenizer = tokenizer,
                              prompt = prompt,
                              max_new_tokens = max_new_tokens,
                              temperature = temperature)
    print(generated_text)
    print()

In [ ]:
# Supervised Finetune 지도 미세조정

import sys
sys.path.append(".")

import json
from codebot.tokenizer import BPETokenizer

tokenizer = BPETokenizer.load_from("codebot/merge_rules.pkl")

with open("codebot/tiny_codes_sft.json") as f:
    data = json.load(f)

item = data[0]
print(item)

# 알파카 포멧
text = f"### Instruction:\n{item["instruction"]}\n\n### Response:\n{item["response"]}<|endoftext|>"
print(text)

ids = tokenizer.encode(text)
print(ids)

In [ ]:
import sys
sys.path.append(".")

from itertools import cycle
import json
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
from tqdm import tqdm
from codebot.model import GPT
from codebot.tokenizer import BPETokenizer
from codebot.utils import get_device

In [ ]:
device = get_device()
print(device)
data_path = "codebot/tiny_codes_sft.json"
tokenizer_path = "codebot/merge_rules.pkl"
pretrain_model_path = "codebot/model_pretrain.pt"
sft_model_save_path = "codebot/model_sft.pt"

context_len = 256
batch_size = 32
learning_rate = 3e-4
max_iters = 500

In [ ]:
class SFTDataset(Dataset):
    def __init__(self, data_path, tokenizer, context_len):
        self.tokenizer = tokenizer
        self.context_len = context_len
        self.samples = []

        with open(data_path) as f:
            data = json.load(f)

        for item in data:
            ids, labels = self._create_sample(item["instruction"], item["response"])
            self.samples.append((ids, labels))

    def _create_sample(self, instruction, response):
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
        response = f"{response}<|endoftext|>"
        
        prompt_ids = self.tokenizer.encode(prompt)
        response_ids = self.tokenizer.encode(response)

        ids = prompt_ids + response_ids
        labels = [-100] * len(prompt_ids) + response_ids

        ids = ids[:-1]
        labels = labels[1:]

        pad_len = self.context_len - len(ids)
        if pad_len > 0:
            ids = ids + [0] * pad_len
            labels = labels + [-100] * pad_len
        elif pad_len < 0:
            ids = ids[:self.context_len]
            labels = labels[:self.context_len]

        return ids, labels

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids, labels = self.samples[idx]
        return torch.tensor(ids, dtype = torch.long), torch.tensor(labels, dtype = torch.long)

In [ ]:
tokenizer = BPETokenizer.load_from(tokenizer_path)
dataset = SFTDataset(data_path, tokenizer, context_len)
dataloader = DataLoader(dataset, batch_size = batch_size, shuffle = True)

model = GPT.load_from(pretrain_model_path, device = device)
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

losses = []
data_iter = cycle(dataloader)

In [ ]:
pbar = tqdm(range(max_iters))

for i in pbar:
    batch_x, batch_y = next(data_iter)
    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    logits = model(batch_x)
    loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                           batch_y.view(-1),
                           ignore_index = -100)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    pbar.set_postfix({"loss": f"{loss.item():.4f}"})

plt.figure(figsize = (10, 6))
plt.plot(losses)
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.grid(True)
plt.savefig("loss_sft.png")

model.save(sft_model_save_path)

In [ ]:
import sys
sys.path.append(".")

from codebot.model import GPT
from codebot.tokenizer import BPETokenizer
from codebot.utils import generate, get_device

device = get_device()
print(device)

model_path = "codebot/model_sft.pt"
tokenizer_path = "codebot/merge_rules.pkl"
max_new_tokens = 200
temperature = 1.0

def format_prompt(user_message):
    return f"### Instruction:\n{user_message}\n\n### Response:\n"

tokenizer = BPETokenizer.load_from(tokenizer_path)
model = GPT.load_from(model_path, device = device)

while True:
    user_input = input("\nYou: ").strip()

    if not user_input:
        continue

    prompt = format_prompt(user_input)
    response = generate(model, tokenizer, prompt, max_new_tokens, temperature)

    if "### Response:" in response:
        response = response.split("### Response:")[-1].strip()

    if "\n" in response:
        print(f"Bot:\n{response}")
    else:
        print(f"Bot: {response}")